# Olist Marketplace Delivery Operations Analysis

This notebook reproduces the core delivery KPIs. Place the nine source CSV files in `../data/raw/` or update `DATA`.

In [ ]:
import pandas as pd
from pathlib import Path
DATA = Path('../data/raw')
orders = pd.read_csv(DATA/'olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date'])
customers = pd.read_csv(DATA/'olist_customers_dataset.csv')
reviews = pd.read_csv(DATA/'olist_order_reviews_dataset.csv')
print(orders.shape, customers.shape, reviews.shape)

## Delivery KPI calculations
Only delivered orders with an actual delivery timestamp are included in delivery-time and lateness metrics.

In [ ]:
review = reviews.groupby('order_id', as_index=False).agg(review_score=('review_score','mean'))
d = orders.query("order_status == 'delivered'").merge(customers[['customer_id','customer_state']],on='customer_id',how='left').merge(review,on='order_id',how='left').dropna(subset=['order_delivered_customer_date']).copy()
d['delivery_days']=(d.order_delivered_customer_date-d.order_purchase_timestamp).dt.total_seconds()/86400
d['delay_days']=(d.order_delivered_customer_date-d.order_estimated_delivery_date).dt.total_seconds()/86400
d['late_delivery']=d.delay_days>0
print('Delivered orders:',d.order_id.nunique())
print('Late delivery rate (%):',round(d.late_delivery.mean()*100,2))
print('Average delivery days:',round(d.delivery_days.mean(),2))
print('Average review score:',round(d.review_score.mean(),2))

In [ ]:
d.groupby('customer_state').agg(orders=('order_id','nunique'),late_rate_pct=('late_delivery',lambda x:x.mean()*100),avg_delivery_days=('delivery_days','mean'),avg_review=('review_score','mean')).query('orders >= 100').sort_values('late_rate_pct',ascending=False).head(15)